# 01 Data Cleaning

This notebook builds the reproducible transaction layer used by the retention analysis.

Outputs:

- `../data/processed/clean_transactions.parquet`
- `../data/processed/data_audit_summary.csv`


## Setup

By default, the notebook expects the raw UCI Online Retail workbook at `../data/raw/Online Retail.xlsx`. Set `RAW_PATH` to use another supported source file and `PROCESSED_DIR` to write outputs to a versioned folder.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PATH = Path(os.environ.get("RAW_PATH", "../data/raw/Online Retail.xlsx"))
PROCESSED_DIR = Path(os.environ.get("PROCESSED_DIR", "../data/processed"))

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw source file not found: {RAW_PATH}. "
        "Place the source file in data/raw or set RAW_PATH before running."
    )

## Load Raw Transactions

The raw file is loaded without dropping rows so that audit metrics reconcile back to the source dataset. Duplicate rows are measured, not removed.

In [ ]:
if RAW_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(RAW_PATH)
elif RAW_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(RAW_PATH)
else:
    raise ValueError(f"Unsupported raw file extension: {RAW_PATH.suffix}")

column_aliases = {
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
}
df = df.rename(columns={source: target for source, target in column_aliases.items() if source in df.columns})


required_columns = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}
missing_columns = sorted(required_columns - set(df.columns))

if missing_columns:
    raise ValueError(f"Raw file is missing required columns: {missing_columns}")

print(f"Rows loaded: {len(df):,}")
df.head()

## Build Analytical Flags

The clean transaction layer keeps all source rows and adds explicit flags for finance reconciliation and customer-level retention analysis.

Important distinction:

- `is_positive_purchase` identifies gross purchase behaviour before customer-level filtering.
- `is_valid_order_line` is the stricter customer-analysis flag used downstream, requiring a positive purchase with a non-null `CustomerID`.
- `analytical_revenue` is populated only for valid customer-level order lines.

In [ ]:
clean_transactions = df.copy()

clean_transactions["InvoiceDate"] = pd.to_datetime(clean_transactions["InvoiceDate"])
clean_transactions["line_revenue_gross"] = (
    clean_transactions["Quantity"] * clean_transactions["UnitPrice"]
)

clean_transactions["is_cancelled_invoice"] = (
    clean_transactions["InvoiceNo"].astype(str).str.startswith("C")
)
clean_transactions["is_refund_or_return"] = (
    (clean_transactions["Quantity"] < 0)
    | clean_transactions["is_cancelled_invoice"]
)
clean_transactions["is_positive_purchase"] = (
    (clean_transactions["Quantity"] > 0)
    & (clean_transactions["UnitPrice"] > 0)
    & ~clean_transactions["is_cancelled_invoice"]
)
clean_transactions["is_identified_customer"] = clean_transactions["CustomerID"].notna()

clean_transactions["exclusion_reason"] = np.select(
    [
        clean_transactions["is_cancelled_invoice"],
        clean_transactions["Quantity"] < 0,
        clean_transactions["UnitPrice"] <= 0,
        clean_transactions["CustomerID"].isna(),
    ],
    [
        "cancelled_invoice",
        "negative_quantity",
        "non_positive_unit_price",
        "missing_customer_id",
    ],
    default="valid_order_line",
)

clean_transactions["is_valid_order_line"] = (
    clean_transactions["is_positive_purchase"]
    & clean_transactions["is_identified_customer"]
)
clean_transactions["analytical_revenue"] = np.where(
    clean_transactions["is_valid_order_line"],
    clean_transactions["line_revenue_gross"],
    0.0,
)

clean_transactions.head()

## Audit Summary

The audit separates finance-style revenue reconciliation from customer-identifiable analytical revenue. Lifecycle and retention notebooks use identifiable customer revenue only.

In [ ]:
total_rows = len(clean_transactions)
duplicate_rows = clean_transactions.duplicated().sum()
missing_customer_pct = clean_transactions["CustomerID"].isna().mean() * 100

gross_purchase_revenue = clean_transactions.loc[
    clean_transactions["is_positive_purchase"],
    "line_revenue_gross",
].sum()
cancelled_refund_revenue = clean_transactions.loc[
    clean_transactions["is_refund_or_return"],
    "line_revenue_gross",
].sum()
net_transactional_revenue = gross_purchase_revenue + cancelled_refund_revenue

analytical_purchase_revenue = gross_purchase_revenue
identifiable_customer_revenue = clean_transactions.loc[
    clean_transactions["is_valid_order_line"],
    "line_revenue_gross",
].sum()
identifiable_revenue_pct = (
    identifiable_customer_revenue / analytical_purchase_revenue
) * 100

valid_customers = clean_transactions.loc[
    clean_transactions["is_valid_order_line"],
    "CustomerID",
].nunique()
valid_orders = clean_transactions.loc[
    clean_transactions["is_valid_order_line"],
    "InvoiceNo",
].nunique()

audit_summary = pd.DataFrame(
    {
        "metric": [
            "total_rows",
            "duplicate_rows",
            "missing_customer_pct",
            "gross_purchase_revenue",
            "cancelled_refund_revenue",
            "net_transactional_revenue",
            "analytical_purchase_revenue",
            "identifiable_customer_revenue",
            "identifiable_revenue_pct",
            "valid_customers",
            "valid_orders",
        ],
        "value": [
            total_rows,
            duplicate_rows,
            missing_customer_pct,
            gross_purchase_revenue,
            cancelled_refund_revenue,
            net_transactional_revenue,
            analytical_purchase_revenue,
            identifiable_customer_revenue,
            identifiable_revenue_pct,
            valid_customers,
            valid_orders,
        ],
    }
)

audit_summary

## Save Processed Outputs

The parquet output keeps the original source column names plus the analytical fields expected by the retention notebook. Mixed-type object columns are cast before saving for stable parquet serialization.

In [ ]:
clean_transactions["InvoiceNo"] = clean_transactions["InvoiceNo"].astype(str)
clean_transactions["StockCode"] = clean_transactions["StockCode"].astype(str)
clean_transactions["Description"] = clean_transactions["Description"].astype("string")
clean_transactions["Country"] = clean_transactions["Country"].astype("string")
clean_transactions["exclusion_reason"] = clean_transactions["exclusion_reason"].astype("string")
clean_transactions["CustomerID"] = clean_transactions["CustomerID"].astype("Int64")

clean_transactions.to_parquet(
    PROCESSED_DIR / "clean_transactions.parquet",
    index=False,
)
audit_summary.to_csv(
    PROCESSED_DIR / "data_audit_summary.csv",
    index=False,
)

print(f"Saved {len(clean_transactions):,} rows to {PROCESSED_DIR / 'clean_transactions.parquet'}")
print(f"Saved audit summary to {PROCESSED_DIR / 'data_audit_summary.csv'}")

## Read-Back Validation

This final check confirms the saved parquet includes the fields required by downstream retention analysis.

In [ ]:
required_processed_columns = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
    "line_revenue_gross",
    "is_cancelled_invoice",
    "is_refund_or_return",
    "is_positive_purchase",
    "is_identified_customer",
    "exclusion_reason",
    "is_valid_order_line",
    "analytical_revenue",
}

saved_transactions = pd.read_parquet(PROCESSED_DIR / "clean_transactions.parquet")
missing_processed_columns = sorted(
    required_processed_columns - set(saved_transactions.columns)
)

if missing_processed_columns:
    raise ValueError(
        f"Processed output is missing required columns: {missing_processed_columns}"
    )

print(f"Validated processed rows: {len(saved_transactions):,}")
saved_transactions.head()